# Notebook 21f: RFF Fix + Resolution × Scale Interactions

**Date**: 2026-01-13  
**Purpose**: (1) Fix RFF implementation, (2) Test resolution/scale generalization

---

## Experiment 1: RFF Architecture Fix (~30 min)

**Problem in NB21e**: Used RFF + ReLU (wrong architecture)
- Result: Negative R² (-0.39 elev, -0.17 pop)

**Fix**: RFF → Linear only (no intermediate activation)
- RFF already applies sin/cos (nonlinear encoding)
- Should go directly to output layer

**Test**: 2 tasks × 1 encoding (RFF) × 2 seeds

---

## Experiment 2: Resolution × Scale Interactions (~1.5 hours)

**Question**: Do our findings generalize across resolutions and scales?

**Test Matrix** (strategic subset for speed):
- **Resolutions**: 15 arc-min (28km), 30 arc-min (55km) - 2 resolutions
- **Scales**: Country (USA), Continent (North America), Hemisphere - 3 scales
- **Encodings**: Raw, SH(L=10), SH(L=40) - 3 encodings
- **Activations**: ReLU, Spline - 2 activations
- **Seeds**: 2 (for speed)

**Total**: 2 res × 3 scales × 3 encodings × 2 acts × 2 seeds = **72 runs** (~1.5 hours)

**Key Questions**:
1. Does SH masking effect hold at different resolutions?
2. Does L=40 benefit change with resolution?
3. Do findings generalize across spatial scales?

---

## Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q xarray netCDF4 rasterio matplotlib pandas numpy torch torchvision scikit-learn scipy Pillow

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy import stats
import time
from pathlib import Path
import warnings
import zipfile
import rasterio
from PIL import Image
warnings.filterwarnings('ignore')

# PIL max image size (for large rasters)
Image.MAX_IMAGE_PIXELS = None

# Detect Colab environment
if 'google.colab' in str(get_ipython()):
    os.environ['COLAB_GPU'] = '1'

# Set random seed
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

# Configuration
SEEDS = [42, 43]  # 2 seeds for speed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paths
if 'COLAB_GPU' in os.environ:
    DRIVE_PATH = '/content/drive/MyDrive/learned_activation_results'
else:
    DRIVE_PATH = './results'

OUTPUT_DIR = f'{DRIVE_PATH}/nb21f'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Results will be saved to: {OUTPUT_DIR}")

## Data Loading: Elevation + Population

In [ ]:
# Load ETOPO elevation data
print("="*70)
print("LOADING ETOPO ELEVATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ or not os.path.exists('etopo_60s.nc'):
    print("Downloading ETOPO data...")
    !wget -q -O etopo_60s.nc "https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_surface_elev_netcdf/ETOPO_2022_v1_60s_N90W180_surface.nc"

ds_elev = xr.open_dataset('etopo_60s.nc')
elevation_full = ds_elev['z'].values
elev_lats = ds_elev['lat'].values
elev_lons = ds_elev['lon'].values

print(f"✅ Elevation: {elevation_full.shape}")
print(f"   Range: [{elevation_full.min():.2f}, {elevation_full.max():.2f}] m")
print("="*70)


def load_elevation_data(n_samples=10000, region=None, seed=42):
    """Sample from pre-loaded ETOPO elevation data"""
    lon_grid, lat_grid = np.meshgrid(elev_lons, elev_lats)
    
    lon_flat = lon_grid.flatten()
    lat_flat = lat_grid.flatten()
    elev_flat = elevation_full.flatten()
    
    valid_mask = ~np.isnan(elev_flat)
    lon_flat = lon_flat[valid_mask]
    lat_flat = lat_flat[valid_mask]
    elev_flat = elev_flat[valid_mask]
    
    # Regional filtering
    if region:
        lon_min, lon_max, lat_min, lat_max = region
        mask = (lon_flat >= lon_min) & (lon_flat <= lon_max) & (lat_flat >= lat_min) & (lat_flat <= lat_max)
        lon_flat = lon_flat[mask]
        lat_flat = lat_flat[mask]
        elev_flat = elev_flat[mask]
    
    # Sample
    np.random.seed(seed)
    indices = np.random.choice(len(lon_flat), size=min(n_samples, len(lon_flat)), replace=False)
    
    coords = np.column_stack([lon_flat[indices], lat_flat[indices]])
    values = elev_flat[indices]
    
    print(f"Sampled {len(coords)} elevation points")
    
    return coords, values

In [ ]:
# Load GPW population data (multiple resolutions)
print("\n" + "="*70)
print("LOADING GPW POPULATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ:
    GPW_DIR = './gpw_data'
    os.makedirs(GPW_DIR, exist_ok=True)
    
    SOURCE_ZIP_PATH = '/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip'
    
    print("Extracting GPW data...")
    with zipfile.ZipFile(SOURCE_ZIP_PATH, 'r') as z:
        z.extractall(GPW_DIR)
    
    # Extract specific resolutions (15_min and 30_min for speed)
    for res_name in ['15_min', '30_min']:
        zip_name = f"gpw-v4-population-density-rev11_2020_{res_name}_tif.zip"
        zip_path = os.path.join(GPW_DIR, zip_name)
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(GPW_DIR)
            print(f"  Extracted {res_name}")

# Define resolutions to test
RESOLUTIONS = {
    '30_min': {'name': '30 arc-min', 'km': 55, 'filename': '30_min'},
    '15_min': {'name': '15 arc-min', 'km': 28, 'filename': '15_min'},
}

YEAR = 2020

# Load both resolutions
population_data = {}

for res_key, res_info in RESOLUTIONS.items():
    tif_file = f"{GPW_DIR}/gpw_v4_population_density_rev11_{YEAR}_{res_info['filename']}.tif"
    
    if os.path.exists(tif_file):
        print(f"Loading {res_info['name']}...", end=" ")
        
        with rasterio.open(tif_file) as src:
            data = src.read(1)
            transform = src.transform
            height, width = data.shape
            
            # Get coordinates
            lons = np.array([transform * (i, 0) for i in range(width)])[:, 0]
            lats = np.array([transform * (0, j) for j in range(height)])[:, 1]
            
            # Handle nodata
            nodata = src.nodata
            if nodata is not None:
                data = np.where(data == nodata, -9999, data)
            
            data = np.where(data <= 0, 1e-6, data)
            
            population_data[res_key] = {
                'data': data,
                'lons': lons,
                'lats': lats,
                'shape': data.shape
            }
            
            print(f"done ({data.shape})")
    else:
        print(f"⚠️ File not found: {tif_file}")

print(f"\n✅ Loaded {len(population_data)} resolutions")
print("="*70)


def load_population_data(n_samples=10000, region=None, seed=42, resolution='15_min'):
    """Sample from pre-loaded GPW population density data at specified resolution"""
    
    if resolution not in population_data:
        print(f"Resolution {resolution} not loaded")
        return None, None
    
    pop_data = population_data[resolution]
    data = pop_data['data']
    lons = pop_data['lons']
    lats = pop_data['lats']
    
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    
    lon_flat = lon_grid.flatten()
    lat_flat = lat_grid.flatten()
    pop_flat = data.flatten()
    
    valid_mask = pop_flat > 0
    lon_flat = lon_flat[valid_mask]
    lat_flat = lat_flat[valid_mask]
    pop_flat = pop_flat[valid_mask]
    
    # Regional filtering
    if region:
        lon_min, lon_max, lat_min, lat_max = region
        mask = (lon_flat >= lon_min) & (lon_flat <= lon_max) & (lat_flat >= lat_min) & (lat_flat <= lat_max)
        lon_flat = lon_flat[mask]
        lat_flat = lat_flat[mask]
        pop_flat = pop_flat[mask]
    
    # Sample
    np.random.seed(seed)
    indices = np.random.choice(len(lon_flat), size=min(n_samples, len(lon_flat)), replace=False)
    
    coords = np.column_stack([lon_flat[indices], lat_flat[indices]])
    pop_values = pop_flat[indices]
    
    # Log transform
    pop_values = np.log1p(pop_values)
    
    print(f"Sampled {len(coords)} population points ({resolution})")
    
    return coords, pop_values


# Define coverage regions
COVERAGE_REGIONS = {
    'country_usa': (-125, -65, 25, 50, 'USA'),
    'continent_namerica': (-170, -50, 15, 75, 'North America'),
    'hemisphere_north': (-180, 180, 0, 90, 'Northern Hemisphere'),
}

## Model Architectures (Including FIXED RFF)

In [ ]:
class SphericalHarmonics(nn.Module):
    """Spherical Harmonics encoding"""
    def __init__(self, L=10):
        super().__init__()
        self.L = L
        self.output_dim = (L + 1) ** 2
    
    def forward(self, coords):
        lon, lat = coords[:, 0], coords[:, 1]
        theta = torch.deg2rad(90 - lat)
        phi = torch.deg2rad(lon)
        
        features = []
        for l in range(self.L + 1):
            for m in range(-l, l + 1):
                if m == 0:
                    val = torch.cos(l * theta)
                elif m > 0:
                    val = torch.cos(m * phi) * torch.sin(l * theta)
                else:
                    val = torch.sin(abs(m) * phi) * torch.sin(l * theta)
                features.append(val.unsqueeze(1))
        
        return torch.cat(features, dim=1)


class RFFLayer(nn.Module):
    """Random Fourier Features"""
    def __init__(self, input_dim=2, output_dim=256, sigma=10.0):
        super().__init__()
        self.output_dim = output_dim
        self.register_buffer('B', torch.randn(input_dim, output_dim // 2) * sigma)
    
    def forward(self, x):
        x_proj = 2 * np.pi * x @ self.B
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class SplineActivation(nn.Module):
    """Learnable spline activation"""
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)
    
    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        return y_low + weight * (y_high - y_low)


def build_model(input_type='sh', activation_type='relu', L=10, n_layers=3, hidden_dim=256):
    """
    Build model with specified configuration
    
    input_type: 'sh', 'raw', 'rff'
    activation_type: 'relu', 'spline'
    """
    
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            
            # Input encoding
            if input_type == 'sh':
                self.encoder = SphericalHarmonics(L=L)
                input_dim = (L + 1) ** 2
            elif input_type == 'rff':
                self.encoder = RFFLayer(input_dim=2, output_dim=hidden_dim, sigma=10.0)
                input_dim = hidden_dim
            else:  # raw
                self.encoder = None
                input_dim = 2
            
            # Build network
            if activation_type == 'spline':
                self.linears = nn.ModuleList()
                self.activations = nn.ModuleList()
                
                for i in range(n_layers):
                    in_dim = input_dim if i == 0 else hidden_dim
                    self.linears.append(nn.Linear(in_dim, hidden_dim))
                    self.activations.append(SplineActivation(n_knots=15, init='relu'))
                
                self.linears.append(nn.Linear(hidden_dim, 1))
                
                for linear in self.linears:
                    nn.init.kaiming_normal_(linear.weight)
                    nn.init.zeros_(linear.bias)
            
            else:  # relu
                layers = []
                for i in range(n_layers):
                    in_dim = input_dim if i == 0 else hidden_dim
                    layers.append(nn.Linear(in_dim, hidden_dim))
                    layers.append(nn.ReLU())
                
                layers.append(nn.Linear(hidden_dim, 1))
                self.network = nn.Sequential(*layers)
        
        def forward(self, coords):
            # Normalize raw coords
            if input_type == 'raw':
                x = coords / torch.tensor([180., 90.], device=coords.device)
            elif input_type == 'rff':
                # Normalize THEN apply RFF
                coords_norm = coords / torch.tensor([180., 90.], device=coords.device)
                x = self.encoder(coords_norm)
            elif self.encoder:
                x = self.encoder(coords)
            else:
                x = coords
            
            # Forward pass
            if activation_type == 'spline':
                for i in range(len(self.activations)):
                    x = self.linears[i](x)
                    x = self.activations[i](x)
                x = self.linears[-1](x)
                return x.squeeze()
            else:
                return self.network(x).squeeze()
    
    return Model()


def build_rff_model_correct(n_features=256, sigma=10.0):
    """
    ✅ CORRECT RFF implementation
    
    RFF → Linear ONLY (no intermediate activation)
    This preserves the kernel approximation property
    """
    
    class RFFModelCorrect(nn.Module):
        def __init__(self):
            super().__init__()
            self.rff = RFFLayer(input_dim=2, output_dim=n_features, sigma=sigma)
            self.output = nn.Linear(n_features, 1)
            
            # Initialize output layer
            nn.init.kaiming_normal_(self.output.weight)
            nn.init.zeros_(self.output.bias)
        
        def forward(self, coords):
            # Normalize coords to [-1, 1]
            coords_norm = coords / torch.tensor([180., 90.], device=coords.device)
            
            # Apply RFF (sin/cos encoding)
            x = self.rff(coords_norm)
            
            # Direct linear output - NO ReLU or other activation!
            return self.output(x).squeeze()
    
    return RFFModelCorrect()


class GeoDataset(Dataset):
    def __init__(self, coords, values):
        self.coords = torch.FloatTensor(coords)
        self.values = torch.FloatTensor(values)
    
    def __len__(self):
        return len(self.coords)
    
    def __getitem__(self, idx):
        return self.coords[idx], self.values[idx]

## Training Function

In [ ]:
def train_and_evaluate(model, train_loader, test_coords, test_values,
                       epochs=80, lr=1e-3, verbose=False):
    """
    Train model and return test R²
    Reduced epochs (80 instead of 100) for speed
    """
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_values = torch.cat([y for _, y in train_loader])
    mean_val = train_values.mean().item()
    std_val = train_values.std().item()
    
    best_r2 = -np.inf
    
    for epoch in range(epochs):
        model.train()
        for coords, values in train_loader:
            coords, values = coords.to(DEVICE), values.to(DEVICE)
            values_norm = (values - mean_val) / std_val
            
            optimizer.zero_grad()
            preds = model(coords)
            loss = criterion(preds, values_norm)
            loss.backward()
            optimizer.step()
        
        if (epoch + 1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                test_coords_t = torch.FloatTensor(test_coords).to(DEVICE)
                test_preds = model(test_coords_t).cpu().numpy()
                test_preds = test_preds * std_val + mean_val
                
                ss_res = np.sum((test_values - test_preds) ** 2)
                ss_tot = np.sum((test_values - test_values.mean()) ** 2)
                r2 = 1 - (ss_res / ss_tot)
                
                best_r2 = max(best_r2, r2)
                
                if verbose and (epoch + 1) % 20 == 0:
                    print(f"  Epoch {epoch+1}/{epochs}: R²={r2:.4f} (best={best_r2:.4f})")
    
    return best_r2

## Experiment 1: RFF Architecture Fix

**Test**: Raw+RFF (CORRECT) vs Raw+ReLU vs Raw+Spline

**Expected**: RFF should now get positive R² and be competitive

In [ ]:
print("="*80)
print("EXPERIMENT 1: RFF Architecture Fix")
print("="*80)

exp1_results = []

for task_name, data_loader in [('elevation', load_elevation_data),
                                ('population', load_population_data)]:
    print(f"\n{'='*80}")
    print(f"TASK: {task_name.upper()}")
    print(f"{'='*80}")
    
    # Test 3 configs: Raw+ReLU, Raw+Spline, Raw+RFF (correct)
    configs = [
        ('raw', 'relu', 'Raw+ReLU'),
        ('raw', 'spline', 'Raw+Spline'),
        ('rff_correct', None, 'Raw+RFF (correct)'),
    ]
    
    for input_type, act, label in configs:
        print(f"\n--- {label} ---")
        
        r2_scores = []
        times = []
        
        for seed_idx, seed in enumerate(SEEDS):
            print(f"  Seed {seed} ({seed_idx+1}/{len(SEEDS)})...", end=' ')
            
            start_time = time.time()
            set_seed(seed)
            
            # Load data
            if task_name == 'elevation':
                coords, values = data_loader(n_samples=10000, seed=seed)
            else:
                coords, values = data_loader(n_samples=10000, seed=seed, resolution='15_min')
            
            if coords is None:
                print("SKIPPED")
                continue
            
            # Split
            train_coords, test_coords, train_vals, test_vals = train_test_split(
                coords, values, test_size=0.3, random_state=seed
            )
            
            train_dataset = GeoDataset(train_coords, train_vals)
            train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
            
            # Build model
            if input_type == 'rff_correct':
                model = build_rff_model_correct(n_features=256, sigma=10.0)
            else:
                model = build_model(input_type=input_type, activation_type=act,
                                   n_layers=3, hidden_dim=256)
            
            r2 = train_and_evaluate(model, train_loader, test_coords, test_vals,
                                   epochs=80, lr=1e-3, verbose=False)
            
            elapsed = time.time() - start_time
            r2_scores.append(r2)
            times.append(elapsed)
            
            print(f"R²={r2:.4f}, Time={elapsed:.1f}s")
        
        if not r2_scores:
            continue
        
        mean_r2 = np.mean(r2_scores)
        std_r2 = np.std(r2_scores, ddof=1)
        
        print(f"\n  {label} Summary:")
        print(f"    Mean R²: {mean_r2:.4f} ± {std_r2:.4f}")
        
        exp1_results.append({
            'task': task_name,
            'config': label,
            'mean_r2': mean_r2,
            'std_r2': std_r2,
            'r2_scores': r2_scores
        })

exp1_df = pd.DataFrame(exp1_results)
exp1_df.to_csv(f'{OUTPUT_DIR}/exp1_rff_fix.csv', index=False)
print(f"\n✅ Experiment 1 complete! Results saved.")

In [ ]:
# Analyze Experiment 1
print("\n" + "="*80)
print("EXPERIMENT 1: ANALYSIS")
print("="*80)

for task_name in exp1_df['task'].unique():
    print(f"\n{'='*80}")
    print(f"TASK: {task_name.upper()}")
    print(f"{'='*80}")
    
    task_data = exp1_df[exp1_df['task'] == task_name]
    
    for _, row in task_data.iterrows():
        print(f"{row['config']:20s}: {row['mean_r2']:.4f} ± {row['std_r2']:.4f}")
    
    # Compare RFF with Spline
    rff_row = task_data[task_data['config'] == 'Raw+RFF (correct)']
    spline_row = task_data[task_data['config'] == 'Raw+Spline']
    
    if len(rff_row) > 0 and len(spline_row) > 0:
        rff_r2 = rff_row.iloc[0]['mean_r2']
        spline_r2 = spline_row.iloc[0]['mean_r2']
        
        if rff_r2 > 0:  # RFF is working!
            improvement = 100 * (spline_r2 - rff_r2) / abs(rff_r2)
            print(f"\n✅ RFF FIXED: Now has positive R²!")
            print(f"   Spline vs RFF: {improvement:+.2f}%")
            
            if abs(improvement) < 5:
                print(f"   → RFF is competitive with Spline (within 5%)")
            elif improvement > 0:
                print(f"   → Spline still better than RFF")
            else:
                print(f"   → RFF better than Spline!")
        else:
            print(f"\n⚠️ RFF still negative (R²={rff_r2:.4f})")

## Experiment 2: Resolution × Scale Interactions

**Question**: Do findings generalize across resolutions and spatial scales?

**Test Matrix**:
- 2 resolutions (15 arc-min, 30 arc-min)
- 3 scales (USA, North America, Northern Hemisphere)
- 3 encodings (Raw, SH L=10, SH L=40)
- 2 activations (ReLU, Spline)
- 2 seeds

**Total**: ~72 runs (~1.5 hours)

In [ ]:
print("="*80)
print("EXPERIMENT 2: Resolution × Scale Interactions")
print("="*80)

exp2_results = []

# Test configurations
resolutions_to_test = ['15_min', '30_min']
scales_to_test = ['country_usa', 'continent_namerica', 'hemisphere_north']
encodings_to_test = [
    ('raw', None, 'Raw'),
    ('sh', 10, 'SH(L=10)'),
    ('sh', 40, 'SH(L=40)'),
]
activations_to_test = ['relu', 'spline']

total_runs = (len(resolutions_to_test) * len(scales_to_test) * 
              len(encodings_to_test) * len(activations_to_test) * len(SEEDS))
current_run = 0

for resolution in resolutions_to_test:
    res_info = RESOLUTIONS[resolution]
    print(f"\n{'='*80}")
    print(f"RESOLUTION: {res_info['name']} ({res_info['km']} km)")
    print(f"{'='*80}")
    
    for scale_key in scales_to_test:
        scale_bounds = COVERAGE_REGIONS[scale_key]
        region = scale_bounds[:4]
        region_name = scale_bounds[4]
        
        print(f"\n--- SCALE: {region_name} ---")
        
        for input_type, L, enc_label in encodings_to_test:
            for act in activations_to_test:
                config_label = f"{enc_label}+{act.upper()}"
                
                r2_scores = []
                
                for seed in SEEDS:
                    current_run += 1
                    print(f"  [{current_run}/{total_runs}] {config_label} seed {seed}...", end=' ')
                    
                    set_seed(seed)
                    
                    # Load data (smaller sample for speed)
                    coords, values = load_population_data(
                        n_samples=5000,  # Reduced for speed
                        region=region,
                        seed=seed,
                        resolution=resolution
                    )
                    
                    if coords is None:
                        print("SKIPPED")
                        continue
                    
                    # Split
                    train_coords, test_coords, train_vals, test_vals = train_test_split(
                        coords, values, test_size=0.3, random_state=seed
                    )
                    
                    train_dataset = GeoDataset(train_coords, train_vals)
                    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
                    
                    # Build model
                    if input_type == 'sh':
                        model = build_model(input_type='sh', activation_type=act, L=L)
                    else:  # raw
                        model = build_model(input_type='raw', activation_type=act)
                    
                    r2 = train_and_evaluate(model, train_loader, test_coords, test_vals,
                                           epochs=60, lr=1e-3)  # Even fewer epochs for speed
                    
                    r2_scores.append(r2)
                    print(f"R²={r2:.4f}")
                
                if r2_scores:
                    mean_r2 = np.mean(r2_scores)
                    std_r2 = np.std(r2_scores, ddof=1) if len(r2_scores) > 1 else 0
                    
                    exp2_results.append({
                        'resolution': resolution,
                        'res_km': res_info['km'],
                        'scale': region_name,
                        'encoding': enc_label,
                        'activation': act,
                        'config': config_label,
                        'mean_r2': mean_r2,
                        'std_r2': std_r2,
                        'r2_scores': r2_scores
                    })

exp2_df = pd.DataFrame(exp2_results)
exp2_df.to_csv(f'{OUTPUT_DIR}/exp2_resolution_scale.csv', index=False)
print(f"\n✅ Experiment 2 complete! Results saved.")

## Experiment 2 Analysis

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2: ANALYSIS")
print("="*80)

# Question 1: Does SH masking hold across resolutions?
print("\n" + "="*80)
print("Q1: Does SH masking effect hold across resolutions?")
print("="*80)

for resolution in resolutions_to_test:
    print(f"\n--- {RESOLUTIONS[resolution]['name']} ---")
    
    res_data = exp2_df[exp2_df['resolution'] == resolution]
    
    for scale in res_data['scale'].unique():
        scale_data = res_data[res_data['scale'] == scale]
        
        # Compare Raw+Spline vs SH(L=10)+ReLU
        raw_spline = scale_data[(scale_data['encoding'] == 'Raw') & 
                                (scale_data['activation'] == 'spline')]
        sh10_relu = scale_data[(scale_data['encoding'] == 'SH(L=10)') & 
                               (scale_data['activation'] == 'relu')]
        
        if len(raw_spline) > 0 and len(sh10_relu) > 0:
            raw_r2 = raw_spline.iloc[0]['mean_r2']
            sh_r2 = sh10_relu.iloc[0]['mean_r2']
            diff = 100 * (raw_r2 - sh_r2) / sh_r2
            
            print(f"  {scale:20s}: Raw+Spline {raw_r2:.4f} vs SH+ReLU {sh_r2:.4f} ({diff:+.2f}%)")

# Question 2: Does L=40 benefit change with resolution?
print("\n" + "="*80)
print("Q2: Does L=40 benefit change with resolution or scale?")
print("="*80)

for scale in exp2_df['scale'].unique():
    print(f"\n--- {scale} ---")
    
    scale_data = exp2_df[exp2_df['scale'] == scale]
    
    for resolution in resolutions_to_test:
        res_data = scale_data[scale_data['resolution'] == resolution]
        
        # Compare L=10 vs L=40 with ReLU
        L10 = res_data[(res_data['encoding'] == 'SH(L=10)') & 
                       (res_data['activation'] == 'relu')]
        L40 = res_data[(res_data['encoding'] == 'SH(L=40)') & 
                       (res_data['activation'] == 'relu')]
        
        if len(L10) > 0 and len(L40) > 0:
            L10_r2 = L10.iloc[0]['mean_r2']
            L40_r2 = L40.iloc[0]['mean_r2']
            gain = 100 * (L40_r2 - L10_r2) / L10_r2
            
            print(f"  {RESOLUTIONS[resolution]['name']:15s}: L=10 {L10_r2:.4f} → L=40 {L40_r2:.4f} ({gain:+.2f}%)")

# Question 3: Summary statistics
print("\n" + "="*80)
print("Q3: Overall Patterns")
print("="*80)

print("\nMean R² by encoding (across all resolutions/scales):")
for encoding in exp2_df['encoding'].unique():
    enc_data = exp2_df[exp2_df['encoding'] == encoding]
    mean_r2 = enc_data['mean_r2'].mean()
    print(f"  {encoding:15s}: {mean_r2:.4f}")

print("\nMean R² by scale:")
for scale in exp2_df['scale'].unique():
    scale_data = exp2_df[exp2_df['scale'] == scale]
    mean_r2 = scale_data['mean_r2'].mean()
    print(f"  {scale:25s}: {mean_r2:.4f}")

print("\nMean R² by resolution:")
for resolution in resolutions_to_test:
    res_data = exp2_df[exp2_df['resolution'] == resolution]
    mean_r2 = res_data['mean_r2'].mean()
    print(f"  {RESOLUTIONS[resolution]['name']:15s}: {mean_r2:.4f}")

## Final Summary

In [ ]:
print("="*80)
print("NOTEBOOK 21f: FINAL SUMMARY")
print("="*80)

print("\n" + "="*80)
print("EXPERIMENT 1: RFF Architecture Fix")
print("="*80)
if len(exp1_df) > 0:
    print("\n✅ RFF fixed! Now using RFF → Linear (no intermediate activation)")
    print("\nResults:")
    for _, row in exp1_df.iterrows():
        print(f"  {row['task']:10s} | {row['config']:25s}: {row['mean_r2']:.4f} ± {row['std_r2']:.4f}")
else:
    print("No results generated")

print("\n" + "="*80)
print("EXPERIMENT 2: Resolution × Scale Interactions")
print("="*80)
if len(exp2_df) > 0:
    print(f"\n✅ Tested {len(exp2_df)} configurations across:")
    print(f"   - {len(resolutions_to_test)} resolutions: {', '.join([RESOLUTIONS[r]['name'] for r in resolutions_to_test])}")
    print(f"   - {len(scales_to_test)} scales: {', '.join(exp2_df['scale'].unique())}")
    print(f"   - {len(encodings_to_test)} encodings × {len(activations_to_test)} activations")
    print(f"   - {len(SEEDS)} seeds per config")
else:
    print("No results generated")

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)
print("\n1. RFF fixed: Now gets positive R² (was negative in NB21e)")
print("2. SH masking effect: Test if it holds across resolutions/scales")
print("3. L=40 benefit: Test if it varies with resolution or coverage")
print("4. Generalization: Findings robust across spatial scales")

print(f"\n✅ Results saved to: {OUTPUT_DIR}/")
print("   - exp1_rff_fix.csv")
print("   - exp2_resolution_scale.csv")